In [0]:
from pyspark.sql import functions as F

new_rows = (
    spark.table("samples.tpch.orders")
    .filter("o_orderdate = '1997-12-30'")
    .withColumn("ingested_at", F.current_timestamp())   # a column the table doesn't have
)

new_rows.write.mode("append").saveAsTable("workspace.bronze.orders_1998")

[DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 0cffd9b6-c85c-439c-b4b5-7c9c2d982b6d).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.


Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TABLE command for changing the schema.

Schema merge is not true, either merge is to be enabled ot need to alter the table to let this run.


In [0]:
from pyspark.sql import functions as F

new_rows = (
    spark.table("samples.tpch.orders")
    .filter("o_orderdate = '1997-12-30'")
    .withColumn("ingested_at", F.current_timestamp())   # a column the table doesn't have
)

(
    new_rows.write
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("workspace.bronze.orders_1998")
)

In [0]:
%sql
SELECT * FROM workspace.bronze.orders_1998 
WHERE ingested_at is NOT NULL
LIMIT 10

In [0]:
from pyspark.sql import functions as F

def table_summary(table_name: str) -> dict:
    """Return a quick health summary for a Delta table."""
    row_count = spark.table(table_name).count()
    history = spark.sql(f"DESCRIBE HISTORY {table_name}")
    latest = history.orderBy(F.desc("version")).first()

    return {
        "table": table_name,
        "rows": row_count,
        "latest_version": latest["version"],
        "last_operation": latest["operation"],
        "last_changed": latest["timestamp"],
        "total_versions": history.count()   # hint: history.count()
    }

tables = [
    "workspace.bronze.orders_1998",
    "workspace.bronze.does_not_exist",
]

for t in tables:
    try:
        print(table_summary(t))
    except Exception as e:
        print({"table": t, "error": str(e)})